# 플래닝과 시스템 프롬프트

0.7에서 플래닝과 기본 프롬프트가 어떻게 달라졌는지 측정 범위와 함께 확인합니다.

## 설치

고정된 패키지를 현재 커널에 설치합니다.

In [1]:
# 실습에서 검증한 패키지 버전을 현재 커널에 설치합니다.
%pip install -q deepagents==0.7.0 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

상위 폴더의 .env를 찾아 키 값은 출력하지 않고 버전만 확인합니다.

### 코드 읽기와 비교 경계

`TodoListMiddleware()`는 모델에게 `write_todos` 도구와 그 사용 지침을 추가합니다. 이를 `middleware=[...]`로 전달하는 것은 모든 요청에 계획을 강제한다는 뜻이 아니라, 모델이 할 일 목록을 기록할 수 있는 기능을 선택적으로 연결하는 것입니다. `write_todos` 호출은 계획 내용을 상태에 남기고 후속 단계가 이를 읽게 합니다. 두 할 일을 요청한 예제에서 도구 호출과 저장된 todo 상태를 함께 확인해야 ‘계획 기능이 연결되었다’고 말할 수 있습니다.

시스템 프롬프트 길이와 작업 품질은 같은 측정값이 아닙니다. 기본 지침을 줄이면 첫 호출 입력은 작아질 수 있지만, 업무에 필요한 제약까지 빠지면 재시도나 잘못된 도구 사용이 늘 수 있습니다. 반대로 모든 짧은 질문에 todo 작성을 요구하면 계획 도구 호출 자체가 불필요한 단계가 됩니다. 플래닝은 여러 산출물, 선후 관계, 중간 검증이 있는 작업에 적용하고, 한 단계 조회에는 직접 도구 호출을 우선합니다.


In [2]:
from importlib.metadata import version
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# 현재 폴더부터 상위 폴더까지 .env를 찾고 값은 출력하지 않습니다.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
print('deepagents', version('deepagents'))
print('langchain', version('langchain'))
assert version('deepagents') == '0.7.0'

deepagents 0.7.0
langchain 1.4.0


## 단계 1

0.7에서 opt-in인 TodoListMiddleware를 에이전트에 추가합니다.

In [3]:
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import TodoListMiddleware
from deepagents import create_deep_agent

# 0.7에서는 플래닝 미들웨어를 직접 추가합니다.
model = init_chat_model('openai:gpt-5.4-mini')
agent = create_deep_agent(model=model, middleware=[TodoListMiddleware()])
print('opt-in 미들웨어:', 'TodoListMiddleware')

opt-in 미들웨어: TodoListMiddleware


## 단계 2

두 할 일을 기록하도록 요청하고 write_todos 호출과 상태를 확인합니다.

In [4]:
# 두 단계 계획을 먼저 기록하도록 요청합니다.
result = agent.invoke({'messages':[{'role':'user','content':'먼저 write_todos로 버전 확인과 결과 설명의 두 할 일을 기록하고, 그 다음 짧게 완료하세요.'}]})
calls = [call for m in result['messages'] for call in getattr(m, 'tool_calls', [])]
print('호출 도구:', [call['name'] for call in calls])
print('todos 상태:', result.get('todos'))
assert any(call['name'] == 'write_todos' for call in calls)
assert result.get('todos')

호출 도구: ['write_todos', 'write_todos']
todos 상태: [{'content': '버전 확인', 'status': 'completed'}, {'content': '결과 설명', 'status': 'completed'}]


## 단계 3

릴리스 PR의 제한된 첫 턴 토큰 수로 감소율을 다시 계산합니다.

In [5]:
# 릴리스 PR의 한정된 첫 턴 비교를 계산합니다.
before, after = 5395, 1895
reduction = (before - after) / before
print('gpt-5.5 hello 첫 턴:', before, '->', after, f'({reduction:.1%})')
assert round(reduction, 2) == 0.65

gpt-5.5 hello 첫 턴: 5395 -> 1895 (64.9%)
